# Stage-count sweep on the frozen magnet crossing

How many Gauss-Legendre stages does the one-step network need across the LHCb magnet, and
what stops it getting better?

The network solves one giant implicit Runge-Kutta step from z = 2648.2 mm to z = 7826.0 mm
in a single shot: instead of a root-finder iterating on the stage states, a 4x50 network
outputs all q stage states plus the endpoint, and the physics loss asks that those outputs
satisfy the Runge-Kutta relations for the LHCb equation of motion. `q` is the number of
stages, and it is the one thing varied here: q = 2, 4, 8, 16, on the same population, the
same leg, the same optimiser protocol, three seeds each, and in both training modes -
the physics loss and its supervised twin trained on precomputed fp64 RK4 labels.

Two error floors sit under any such network and both are plotted beside it:

* **the exact scheme.** `../Simple_first_pass` solved the very same q-stage scheme
  directly with a root-finder in fp64. Its endpoint error against the fp64 RK4 reference
  is the error of the *equations*, quite apart from who solves them. A network trained to
  satisfy those equations cannot do better than that, so this curve is a ceiling on
  achievable accuracy - low q means a coarse scheme and a high ceiling.
* **the straight line.** Ignoring the magnet entirely and continuing in a straight line
  through the same leg. This is what "doing nothing" costs, and it is the same number for
  every q.

This notebook only *loads* what the scripts wrote. Nothing here retrains or re-scores.


In [ ]:
import os
import pandas as pd
from IPython.display import Image, display

RESULTS = "results"
FIGURES = "figures"

summary = pd.read_csv(os.path.join(RESULTS, "summary.csv"))
stages = pd.read_csv(os.path.join(RESULTS, "error_vs_stages.csv"), comment="#")
print("runs recorded:", len(summary), " converged:", int(summary["converged"].sum()))
summary

## The datasets

One dataset per q, all four built by `build_datasets.py` from the same official-sample
population and the same fiducial requirement (the reference trajectory must stay inside
the field map). The split counts must therefore be identical across q - the cut asks a
question about the leg and the field, not about how many planes we sample the leg on -
and the q = 8 file must reproduce the verified `One_step_network_v2` dataset array for
array, because that is the file the baseline result was trained on.

In [ ]:
import json, glob
checks = [json.load(open(p)) for p in sorted(glob.glob(os.path.join(RESULTS, "dataset_check_q*.json")))]
for c in sorted(checks, key=lambda c: c["q"]):
    line = "q=%-3d counts %s  match=%s" % (c["q"], c["counts"], c["counts_match"])
    if "bitwise_equal_to_v2_baseline" in c:
        line += "   bitwise == v2 baseline: %s (%d arrays)" % (
            c["bitwise_equal_to_v2_baseline"], c["bitwise_report"]["n_shared_keys"])
    print(line)

## Error against the number of stages

The table below is the experiment in one place: for each q, the median over the three
seeds and the seed-to-seed spread, for both losses, beside the exact scheme's own error
on the same leg.

In [ ]:
cols = ["q",
        "physics_endpoint_med_um", "physics_endpoint_min_um", "physics_endpoint_max_um",
        "data_endpoint_med_um", "data_endpoint_min_um", "data_endpoint_max_um",
        "scheme_endpoint_med_um", "straight_med_um"]
stages[cols].round(1)

In [ ]:
display(Image(filename=os.path.join(FIGURES, "error_vs_stages.png")))

### Reading the left panel

The green curve is the ceiling: at q = 2 the two-stage scheme is itself about 14 mm away
from the truth on this leg, and at q = 4 about 4.3 mm. Those are enormous errors, and they
belong to the scheme, not to the network - no amount of training can get below them. From
q = 8 the ceiling collapses to a few tens of microns, and from that point on the scheme is
no longer what limits the answer; the network's own approximation error is.

So the curve to watch is where the blue and red curves stop following the green one and
flatten. That flattening level is the network floor, and whether q = 8 and q = 16 share it
is the question this experiment was run to answer.

### Reading the right panel

The endpoint is only one of the q + 1 states the network outputs; the other q are the
interior stage states, which the scheme needs but which nothing downstream ever consumes.
Their error is plotted separately because it is the honest measure of how much work the
network is being asked to do, and it is the quantity that grows with q: at q = 16 the
network has 68 numbers to produce per sample instead of 36.

## Did the runs actually converge?

Every run is trained to a genuine stall - restarts until two consecutive ones each improve
the loss by less than 1% - and then a **confirmation pass** with a fresh optimiser, which
counts as converged only if it re-stalls within two restarts with the endpoint medians
unchanged. A run that has not converged is never pooled into the medians above; it appears
in `summary.csv` with `converged = False` and is excluded from `error_vs_stages.csv`.

In [ ]:
display(Image(filename=os.path.join(FIGURES, "convergence.png")))

In [ ]:
summary.groupby(["q", "mode"]).agg(
    n=("seed", "count"),
    converged=("converged", "sum"),
    restarts_med=("restarts", "median"),
    wall_min_med=("wall_s", lambda s: round(s.median() / 60, 1)),
    final_loss_med=("final_loss", "median"),
)

## The comparison that anchors everything

The q = 8 physics runs here are, by construction, the `One_step_network_v2` experiment
run again through the shared driver: the same dataset arrays, the same protocol, the same
three seeds. The only difference is one thread instead of four, which changes the last
bits of every reduction and so cannot be expected to reproduce the numbers exactly - only
the range. v2's converged physics runs landed at 177-235 um, its data twin at 162-208 um.

In [ ]:
q8 = summary[summary["q"] == 8][["mode", "seed", "test_endpoint_med_um", "restarts", "converged"]]
print("v2 baseline: physics 177-235 um, data twin 162-208 um")
q8.round(1)